# Olist E-Commerce - Late Delivery Prediction Project

## 1. Setup & Imports

In [17]:
import pandas as pd 
import sqlite3 
import os  

## 2. Load CSV Files into SQLite Database

In [18]:
conn = sqlite3.connect("olist.db")

In [19]:
def load_csv_to_db(path,conn,name): 
    df_temp= pd.read_csv(path)
    df_temp.to_sql(name, conn, if_exists="replace", index=False)

In [20]:
files_to_tables = {
    "dataSet\\olist_orders_dataset.csv": "orders",
    "dataSet\\olist_customers_dataset.csv": "customers",
    "dataSet\\olist_products_dataset.csv": "products",
    "dataSet\\olist_sellers_dataset.csv": "sellers",
    "dataSet\\olist_order_items_dataset.csv": "order_items",
    "dataSet\\product_category_name_translation.csv": "product_category_name_translation",
    "dataSet\\olist_order_reviews_dataset.csv": "order_reviews",
    "dataSet\\olist_order_payments_dataset.csv": "order_payments",
    "dataSet\\olist_geolocation_dataset.csv": "geolocation"
}
for x,y in files_to_tables.items() :
    load_csv_to_db(x,conn,y)

## 3. Verify Data Integrity (Row Counts Check)

In [21]:
def verify_table(path, conn, name):
    df_csv = pd.read_csv(path)
    csv_rows = len(df_csv)
    
    db_rows = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {name}", conn).iloc[0]['cnt']
    
    match = "✅" if csv_rows == db_rows else "❌"
    print(f"{match} {name}: CSV={csv_rows} rows | DB={db_rows} rows")

for x, y in files_to_tables.items():
    verify_table(x, conn, y)

✅ orders: CSV=99441 rows | DB=99441 rows
✅ customers: CSV=99441 rows | DB=99441 rows
✅ products: CSV=32951 rows | DB=32951 rows
✅ sellers: CSV=3095 rows | DB=3095 rows
✅ order_items: CSV=112650 rows | DB=112650 rows
✅ product_category_name_translation: CSV=71 rows | DB=71 rows
✅ order_reviews: CSV=99224 rows | DB=99224 rows
✅ order_payments: CSV=103886 rows | DB=103886 rows
✅ geolocation: CSV=1000163 rows | DB=1000163 rows


## 4. Exploratory Data Analysis (EDA) with SQL

### 4.1 Orders Table

In [22]:
query = """
SELECT COUNT(*) as null_delivered
FROM orders
WHERE order_delivered_customer_date IS NULL
"""

result = pd.read_sql(query, conn)
print(result)

   null_delivered
0            2965


In [23]:
query = """
SELECT order_status, COUNT(*) as cnt
FROM orders
WHERE order_delivered_customer_date IS NULL
GROUP BY order_status
ORDER BY cnt DESC
"""

result = pd.read_sql(query, conn)
print(result)

  order_status   cnt
0      shipped  1107
1     canceled   619
2  unavailable   609
3     invoiced   314
4   processing   301
5    delivered     8
6      created     5
7     approved     2


In [24]:
query = """
SELECT COUNT(DISTINCT customer_unique_id) as unique_customers,
       COUNT(*) as total_orders
FROM customers
"""

result = pd.read_sql(query, conn)
print(result)

   unique_customers  total_orders
0             96096         99441


In [25]:
query = """
SELECT COUNT(DISTINCT order_id) as unique_orders,
       COUNT(*) as total_items
FROM order_items
"""

result = pd.read_sql(query, conn)
print(result)

   unique_orders  total_items
0          98666       112650


In [26]:
query = """
SELECT order_id, COUNT(*) as num_items
FROM order_items
GROUP BY order_id
ORDER BY num_items DESC
LIMIT 5
"""

result = pd.read_sql(query, conn)
print(result)

                           order_id  num_items
0  8272b63d03f5f79c56e9e4120aec44ef         21
1  ab14fdcfbe524636d65ee38360e22ce8         20
2  1b15974a0141d54e36626dca3fdc731a         20
3  9ef13efd6949e4573a18964dd1bbe7f5         15
4  428a2f660dc84138d969ccd69a0ab6d5         15


In [27]:
query = """
SELECT payment_type, COUNT(*) as cnt
FROM order_payments
GROUP BY payment_type
ORDER BY cnt DESC
"""

result = pd.read_sql(query, conn)
print(result)

  payment_type    cnt
0  credit_card  76795
1       boleto  19784
2      voucher   5775
3   debit_card   1529
4  not_defined      3


In [28]:
query = """
SELECT COUNT(DISTINCT order_id) as unique_orders,
       COUNT(*) as total_reviews
FROM order_reviews
"""
result = pd.read_sql(query, conn)
print(result)

   unique_orders  total_reviews
0          98673          99224


In [29]:
query = """
SELECT COUNT(*) as null_category
FROM products
WHERE product_category_name IS NULL
"""
result = pd.read_sql(query, conn)
print(result)

   null_category
0            610


In [30]:
query_items_agg = """
SELECT 
    order_id,
    COUNT(*) as num_items,
    SUM(price) as total_price,
    SUM(freight_value) as total_freight
FROM order_items
GROUP BY order_id
"""

items_agg = pd.read_sql(query_items_agg, conn)
print(items_agg.head())
print(items_agg.shape)

                           order_id  num_items  total_price  total_freight
0  00010242fe8c5a6d1ba2dd792cb16214          1        58.90          13.29
1  00018f77f2f0320c557190d7a144bdd3          1       239.90          19.93
2  000229ec398224ef6ca0657da4fc703e          1       199.00          17.87
3  00024acbcdf0a6daa1e931b038114c75          1        12.99          12.79
4  00042b26cf59d7ce69dfabb4e55b4fd9          1       199.90          18.14
(98666, 4)


### 4.2 Customers Table

### 4.3 Order Items Table

### 4.4 Payments Table

### 4.5 Reviews Table

### 4.6 Products Table

### 4.7 Sellers Table

### 4.8 Geolocation Table

## 5. Build ML-Ready Table (Aggregation + Joins)

### 5.1 Define Target Variable (Late Delivery)

### 5.2 Aggregate Order Items

### 5.3 Aggregate Payments

### 5.4 Final Join

## 6. Feature Engineering

## 7. Model Training & Evaluation

## Task 1 - Step 3: Test the Database (Join Examples)

In [31]:
query = """
SELECT o.order_id, o.order_status, oi.seller_id, s.seller_city
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN sellers s ON oi.seller_id = s.seller_id
LIMIT 10
"""

result = pd.read_sql(query, conn)
print(result)

                           order_id order_status  \
0  e481f51cbdc54678b7cc49136f2d6af7    delivered   
1  53cdb2fc8bc7dce0b6741e2150273451    delivered   
2  47770eb9100c2d0c44946d9cf07ec65d    delivered   
3  949d5b44dbf5de918fe9c16f97b45f8a    delivered   
4  ad21c59c0840e6cb83a9ceb5573f8159    delivered   
5  a4591c265e18cb1dcee52889e2d8acc3    delivered   
6  136cce7faa42fdb2cefd53fdc79a6098     invoiced   
7  6514b8ad8028c9f2cc2374ded245783f    delivered   
8  76c6e866289321a7c93b82b54852dc33    delivered   
9  e69bfb5eb88e0ed6a785585b27e16dbf    delivered   

                          seller_id            seller_city  
0  3504c0cb71d7fa48d967e0e4c94d59d9                   maua  
1  289cdb325fb7e7f891c38608bf9e0962         belo horizonte  
2  4869f7a5dfa277a7dca6462dcf3b52b2                guariba  
3  66922902710d126a0e7d26b0e3805106         belo horizonte  
4  2c9e548be18521d1c43cde1c582c6de8        mogi das cruzes  
5  8581055ce74af1daba164fdbd55a40de              guarulhos  
